# 02 — Comparaison dictionnaire nettoyé vs non nettoyé

**Objectif :** mesurer l'impact du nettoyage du dictionnaire sur MEDCON.

- `merged_mesh_snomed_dico.json` — dictionnaire brut (445 186 entrées)
- `cleaned_mesh_snomed_dico.json` — dictionnaire nettoyé (2 515 entrées)

**Fichiers nécessaires à uploader :**
- `merged_mesh_snomed_dico.json`
- `cleaned_mesh_snomed_dico.json`
- `corpus_WMT24.json`

## 0. Setup

In [ ]:
import os
os.chdir('/content/Evaluating_medical_machine_translation')
print(f'Répertoire : {os.getcwd()}')

In [ ]:
# Uniquement si pas déjà fait dans la session
from google.colab import files

os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Uploader : merged_mesh_snomed_dico.json + cleaned_mesh_snomed_dico.json + corpus_WMT24.json
uploaded = files.upload()
for fname in uploaded:
    dest = f'data/{fname}'
    os.replace(fname, dest)
    print(f'  → {dest}')

## 1. Imports

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from collections import Counter
from scipy.stats import mannwhitneyu, wilcoxon

os.chdir('/content/Evaluating_medical_machine_translation')
sys.path.insert(0, 'src')
from medcon import build_pairs, build_automaton, medcon_grouped

print('Imports OK')

## 2. Chargement des deux dictionnaires + corpus

In [ ]:
with open('data/cleaned_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict_cleaned = json.load(f)

with open('data/merged_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict_merged = json.load(f)

with open('data/corpus_WMT24.json', encoding='utf-8') as f:
    test_docs = json.load(f)

print(f'Dictionnaire nettoyé : {len(raw_dict_cleaned):>10,} entrées EN')
print(f'Dictionnaire brut    : {len(raw_dict_merged):>10,} entrées EN')
print(f'Réduction            : {100*(1 - len(raw_dict_cleaned)/len(raw_dict_merged)):.1f}%')
print(f'Corpus WMT24         : {len(test_docs)} documents')

## 3. Construction des automates pour les deux dictionnaires

In [ ]:
print('Construction automates — dictionnaire nettoyé...')
pairs_cleaned  = build_pairs(raw_dict_cleaned)
auto_en_cl     = build_automaton(pairs_cleaned, 'en')
auto_fr_cl     = build_automaton(pairs_cleaned, 'fr')
print(f'  Paires : {len(pairs_cleaned):,}  |  Termes EN : {len(auto_en_cl):,}  |  Termes FR : {len(auto_fr_cl):,}')

print('\nConstruction automates — dictionnaire brut (peut prendre ~1 min)...')
pairs_merged   = build_pairs(raw_dict_merged)
auto_en_mg     = build_automaton(pairs_merged, 'en')
auto_fr_mg     = build_automaton(pairs_merged, 'fr')
print(f'  Paires : {len(pairs_merged):,}  |  Termes EN : {len(auto_en_mg):,}  |  Termes FR : {len(auto_fr_mg):,}')

## 4. MEDCON sur les deux dictionnaires

In [ ]:
def run_medcon(test_docs, pairs, auto_en, auto_fr, label):
    print(f'\nMEDCON — {label}')
    results = []
    for i, doc in enumerate(tqdm(test_docs, desc=label)):
        r = medcon_grouped(
            doc['text_en'],
            doc['gpt_translation'],
            pairs, auto_en, auto_fr
        )
        results.append({
            'doc_id'           : i,
            'medcon_f1'        : r['f1'],
            'medcon_precision' : r['precision'],
            'medcon_recall'    : r['recall'],
            'n_expected'       : r['n_expected'],
            'n_found'          : r['n_found'],
            'n_match'          : r['n_match'],
            'en_concepts'      : r['en_concepts'],
            'matched'          : r['matched'],
            'missed'           : r['missed'],
            'extra'            : r['extra'],
        })
    df = pd.DataFrame(results)
    # Docs avec au moins 1 paire attendue (les seuls interprétables)
    df_valid = df[df['n_expected'] > 0]
    print(f'  Docs avec ≥1 paire attendue : {len(df_valid)}/{len(df)}')
    print(f'  Paires attendues (moy.)     : {df["n_expected"].mean():.2f}')
    print(f'  Precision                   : {df_valid["medcon_precision"].mean():.3f}')
    print(f'  Recall                      : {df_valid["medcon_recall"].mean():.3f}')
    print(f'  F1                          : {df_valid["medcon_f1"].mean():.3f}')
    return df


df_cleaned = run_medcon(test_docs, pairs_cleaned, auto_en_cl, auto_fr_cl, 'Nettoyé')
df_merged  = run_medcon(test_docs, pairs_merged,  auto_en_mg, auto_fr_mg, 'Brut')

## 5. Tableau comparatif

In [ ]:
print('=' * 70)
print('COMPARAISON MEDCON — NETTOYÉ vs BRUT')
print('=' * 70)

for label, df in [('Nettoyé', df_cleaned), ('Brut', df_merged)]:
    df_valid = df[df['n_expected'] > 0]
    print(f"\n  {label}")
    print(f"  {'─'*40}")
    print(f"  Docs avec ≥1 paire attendue : {len(df_valid):>5} / {len(df)}")
    print(f"  Paires attendues (moy.)     : {df['n_expected'].mean():>8.2f}")
    print(f"  Paires trouvées  (moy.)     : {df['n_found'].mean():>8.2f}")
    print(f"  Paires matchées  (moy.)     : {df['n_match'].mean():>8.2f}")
    print(f"  Precision (sur docs valides): {df_valid['medcon_precision'].mean():>8.3f}")
    print(f"  Recall    (sur docs valides): {df_valid['medcon_recall'].mean():>8.3f}")
    print(f"  F1        (sur docs valides): {df_valid['medcon_f1'].mean():>8.3f}")

# Test statistique de Wilcoxon sur les F1 (paired)
print('\n--- Test statistique (Wilcoxon signé-rangé sur F1) ---')
try:
    stat, p = wilcoxon(df_cleaned['medcon_f1'], df_merged['medcon_f1'])
    print(f'  W={stat:.1f}  p={p:.4f}  -> {"différence significative" if p < 0.05 else "pas de différence significative"} (α=0.05)')
except Exception as e:
    print(f'  Test non applicable : {e}')

## 6. Analyse qualitative : termes détectés en plus par le dictionnaire brut

In [ ]:
print('TERMES SUPPLÉMENTAIRES DÉTECTÉS PAR LE DICTIONNAIRE BRUT\n')
print(f"{'Doc':>4}  {'Nettoyé':>8}  {'Brut':>8}  Termes EN supplémentaires")
print('-' * 70)

for i in range(len(test_docs)):
    concepts_cl = set(df_cleaned.iloc[i]['en_concepts'])
    concepts_mg = set(df_merged.iloc[i]['en_concepts'])
    extra = concepts_mg - concepts_cl
    n_cl = len(concepts_cl)
    n_mg = len(concepts_mg)
    if extra:
        extra_str = ', '.join(sorted(extra)[:6])
        if len(extra) > 6:
            extra_str += f' ... (+{len(extra)-6})'
        print(f'  {i:>3}  {n_cl:>8}  {n_mg:>8}  {extra_str}')

# Termes supplémentaires les plus fréquents
all_extra_terms = []
for i in range(len(test_docs)):
    concepts_cl = set(df_cleaned.iloc[i]['en_concepts'])
    concepts_mg = set(df_merged.iloc[i]['en_concepts'])
    all_extra_terms.extend(concepts_mg - concepts_cl)

print(f'\nTOP 20 termes EN détectés par le brut mais pas le nettoyé :')
for term, count in Counter(all_extra_terms).most_common(20):
    print(f'  [{count:2d}x] {term}')

## 7. Analyse du bruit : faux positifs dans le dictionnaire nettoyé

In [ ]:
print('FAUX POSITIFS (extras) — NETTOYÉ vs BRUT\n')

for label, df in [('Nettoyé', df_cleaned), ('Brut', df_merged)]:
    all_extra = []
    for el in df['extra']:
        all_extra.extend(el)
    counter = Counter(all_extra)
    print(f'{label} — TOP 15 paires extras (terme FR dans traduction mais EN absent source) :')
    for i, (lbl, count) in enumerate(counter.most_common(15), 1):
        print(f'  {i:2d}. [{count:2d}x] {lbl}')
    print(f'  Total extras uniques : {len(counter)}\n')

## 8. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparaison MEDCON — Dictionnaire nettoyé vs brut (GPT-4o)', fontsize=13, fontweight='bold')

colors = {'Nettoyé': '#3498db', 'Brut': '#e67e22'}

# 1. Distribution F1 — nettoyé
ax = axes[0, 0]
ax.hist(df_cleaned['medcon_f1'], bins=15, color=colors['Nettoyé'], edgecolor='black', alpha=0.85)
ax.axvline(df_cleaned['medcon_f1'].mean(), color='red', linestyle='--', lw=2,
           label=f"Moy. {df_cleaned['medcon_f1'].mean():.3f}")
ax.set_title('Distribution F1 — Nettoyé')
ax.set_xlabel('MEDCON F1')
ax.set_ylabel('Documents')
ax.legend()
ax.grid(alpha=0.3)

# 2. Distribution F1 — brut
ax = axes[0, 1]
ax.hist(df_merged['medcon_f1'], bins=15, color=colors['Brut'], edgecolor='black', alpha=0.85)
ax.axvline(df_merged['medcon_f1'].mean(), color='red', linestyle='--', lw=2,
           label=f"Moy. {df_merged['medcon_f1'].mean():.3f}")
ax.set_title('Distribution F1 — Brut')
ax.set_xlabel('MEDCON F1')
ax.set_ylabel('Documents')
ax.legend()
ax.grid(alpha=0.3)

# 3. Paires attendues par doc — boxplot comparatif
ax = axes[1, 0]
ax.boxplot(
    [df_cleaned['n_expected'], df_merged['n_expected']],
    labels=['Nettoyé', 'Brut'],
    patch_artist=True,
    boxprops=dict(facecolor='#ecf0f1'),
    medianprops=dict(color='red', lw=2)
)
ax.set_title('Paires attendues par document')
ax.set_ylabel('n_expected')
ax.grid(alpha=0.3)

# 4. F1 doc par doc — scatter
ax = axes[1, 1]
ax.scatter(df_cleaned['medcon_f1'], df_merged['medcon_f1'],
           alpha=0.65, color='#8e44ad', edgecolors='white', s=60)
lims = [0, 1]
ax.plot(lims, lims, 'k--', lw=1, alpha=0.5, label='y=x')
ax.set_xlabel('F1 Nettoyé')
ax.set_ylabel('F1 Brut')
ax.set_title('F1 Nettoyé vs F1 Brut (par document)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/02_dict_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée dans results/')

## 9. Export des résultats

In [ ]:
cols_export = ['doc_id', 'medcon_f1', 'medcon_precision', 'medcon_recall',
               'n_expected', 'n_found', 'n_match']

df_cleaned[cols_export].to_json('results/02_medcon_cleaned.json', orient='records', indent=2, force_ascii=False)
df_merged[cols_export].to_json('results/02_medcon_merged.json',  orient='records', indent=2, force_ascii=False)
print('Résultats exportés dans results/')

# Résumé final
df_cl_valid = df_cleaned[df_cleaned['n_expected'] > 0]
df_mg_valid = df_merged[df_merged['n_expected'] > 0]

print('\n' + '=' * 60)
print('RÉSUMÉ FINAL')
print('=' * 60)
print(f"{'':25} {'Nettoyé':>12} {'Brut':>12}")
print('-' * 50)
print(f"{'Entrées dico':25} {len(raw_dict_cleaned):>12,} {len(raw_dict_merged):>12,}")
print(f"{'Docs avec ≥1 paire':25} {len(df_cl_valid):>12} {len(df_mg_valid):>12}")
print(f"{'Paires attendues moy.':25} {df_cleaned['n_expected'].mean():>12.2f} {df_merged['n_expected'].mean():>12.2f}")
print(f"{'F1 moyen (docs valid.)':25} {df_cl_valid['medcon_f1'].mean():>12.3f} {df_mg_valid['medcon_f1'].mean():>12.3f}")